# Horde Roller
Roll a massive number of D20s at once. Set a modifier, a success threshold, and an advantage mode, then see how many hits and misses your horde scores.

In [ ]:
import random

In [ ]:
def roll_horde(num_dice: int, modifier: int = 0, threshold: int = 10, advantage: int = 0) -> dict:
    """
    Roll num_dice D20s, apply modifier to each, and compare against threshold.

    advantage:  1 = advantage     (roll 2d20, keep highest)
                0 = normal        (roll 1d20)
               -1 = disadvantage  (roll 2d20, keep lowest)

    A roll is a SUCCESS if (roll + modifier) >= threshold.
    A natural 20 is always a success. A natural 1 is always a failure.

    Returns a dict with full results.
    """
    if num_dice < 1:
        raise ValueError("Must roll at least 1 die.")
    if advantage not in (-1, 0, 1):
        raise ValueError("advantage must be -1 (disadvantage), 0 (normal), or 1 (advantage).")

    def single_roll():
        if advantage == 0:
            return random.randint(1, 20)
        a, b = random.randint(1, 20), random.randint(1, 20)
        return max(a, b) if advantage == 1 else min(a, b)

    results = []

    for _ in range(num_dice):
        raw = single_roll()
        total = raw + modifier
        if raw == 20:
            success = True   # natural 20 always hits
        elif raw == 1:
            success = False  # natural 1 always fails
        else:
            success = total >= threshold
        results.append({"raw": raw, "total": total, "success": success})

    successes = sum(1 for r in results if r["success"])
    failures  = num_dice - successes
    nat_20s   = sum(1 for r in results if r["raw"] == 20)
    nat_1s    = sum(1 for r in results if r["raw"] == 1)

    return {
        "num_dice":  num_dice,
        "modifier":  modifier,
        "threshold": threshold,
        "advantage": advantage,
        "successes": successes,
        "failures":  failures,
        "nat_20s":   nat_20s,
        "nat_1s":    nat_1s,
        "rolls":     results,
    }

In [ ]:
def print_results(result: dict) -> None:
    """Print a clean summary of a horde roll."""
    adv_label = {1: "Advantage", 0: "Normal", -1: "Disadvantage"}[result["advantage"]]
    sign = "+" if result["modifier"] >= 0 else ""
    print("=" * 40)
    print(f"  HORDE ROLL: {result['num_dice']}d20 {sign}{result['modifier']}")
    print(f"  Mode      : {adv_label}")
    print(f"  Threshold : {result['threshold']}")
    print("=" * 40)
    print(f"  Successes : {result['successes']}")
    print(f"  Failures  : {result['failures']}")
    print(f"  Nat 20s   : {result['nat_20s']}")
    print(f"  Nat 1s    : {result['nat_1s']}")
    pct = result['successes'] / result['num_dice'] * 100
    print(f"  Hit rate  : {pct:.1f}%")
    print("=" * 40)

In [ ]:
# ── Configure your roll here ──────────────────────────────────────
NUM_DICE  = 500   # how many D20s to roll
MODIFIER  = 3     # flat bonus/penalty added to every die (+/- int)
THRESHOLD = 15    # minimum value (after modifier) needed to succeed
ADVANTAGE = 0     # 1 = advantage | 0 = normal | -1 = disadvantage
# ─────────────────────────────────────────────────────────────────

result = roll_horde(NUM_DICE, MODIFIER, THRESHOLD, ADVANTAGE)
print_results(result)

In [ ]:
# Optional: see every individual roll
# Uncomment the lines below if you want the full breakdown.

# for i, r in enumerate(result["rolls"], 1):
#     status = "HIT" if r["success"] else "MISS"
#     print(f"  Die {i:>4}: raw={r['raw']:>2}  total={r['total']:>3}  {status}")